# Classic IP Workshop: Complete Solutions
**IN2040 · Optimization of Organizational Processes · Tecnológico de Monterrey** (Prof. Vincent Boyer)

*Session 4 · after the workshop: the worked example and the 8 problem cards, modeled and solved with PuLP.*

> **How to use this notebook.** Run the **Setup** cell first. Then, for each problem:
> 1. read the **card** and write your own model **before** reading ours;
> 2. compare with **the model** (variables, objective, constraints in plain words);
> 3. run the **PuLP** cell: the same model solved in Python;
> 4. do the **Your turn** changes: remove the twist and watch what happens to the optimum.
>
> The full write-up (class and clue, results checked by hand, why each twist matters) is in the solutions PDF on Canvas.

* Worked example · Concrete for three sites
1. Problem 1 · The snack box
2. Problem 2 · Wi-Fi on campus
3. Problem 3 · The morning routes
4. Problem 4 · Warehouses for Monterrey
5. Problem 5 · The innovation fund
6. Problem 6 · Interns and projects
7. Problem 7 · The night premium
8. Problem 8 · Loading the vans

In [ ]:
# @title ⚙️ Setup: run this cell first (no need to read the code)
import importlib, importlib.metadata, importlib.util, subprocess, sys
try:                                   # PuLP 4 + its CBC solver (pulp[cbc]); replaces an older PuLP 3
    _old = int(importlib.metadata.version("pulp").split(".")[0]) < 4 or importlib.util.find_spec("cbcbox") is None
except importlib.metadata.PackageNotFoundError:
    _old = True
if _old:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pulp[cbc]>=4"], check=True)
import pulp
print("Ready: PuLP", pulp.__version__)

---
## Worked example · Concrete for three sites
**Class:** Transportation, with whole trucks (integer transportation)

### The card
A ready-mix company supplies concrete to 3 building sites from 2 plants, North and South. Concrete travels in mixer trucks that carry 8 m³ each; a trip costs the same whether the truck is full or not. Each plant can dispatch a limited number of truck trips per day, and the road from South to site S3 crosses a bridge that allows at most 2 truck trips per day. How many trucks should go from each plant to each site so that every site gets the concrete it needs at minimum cost?

| MXN per trip | S1 | S2 | S3 | Trips per day |
|---|---|---|---|---|
| North | 900 | 1200 | 1500 | 6 |
| South | 1400 | 1000 | 800 | 7 |
| Need (m³) | 30 | 45 | 20 |  |

### The model
* **Sets:** plants $i\in\{\text{North},\text{South}\}$, sites $j\in\{S1,S2,S3\}$. **Data:** $c_{ij}$ cost of one trip, $T_i$ trips per day, $d_j$ m³ needed.
* **Variables:** $x_{ij}\in\mathbb{Z}_{\ge 0}$ = number of truck trips from plant $i$ to site $j$.
* **Objective:** $\min \sum_i\sum_j c_{ij}\,x_{ij}$ (MXN per day).

| | Constraint | In words |
|---|---|---|
| C1 | $\sum_j x_{ij} \le T_i \quad \forall i$ | a plant cannot send more trips than it has |
| C2 | $8\sum_i x_{ij} \ge d_j \quad \forall j$ | each site gets at least the m³ it needs |
| C3 | $x_{\text{South},S3} \le 2$ | bridge limit |
| C4 | $x_{ij}\in\{0,1,2,\dots\}$ | whole trucks |

In [ ]:
import pulp

plants = ["North", "South"]
sites = ["S1", "S2", "S3"]
trucks_per_day = {"North": 6, "South": 7}            # plant capacity (truck trips per day)
need_m3 = {"S1": 30, "S2": 45, "S3": 20}             # concrete each site needs (m3)
TRUCK_M3 = 8                                         # one truck carries 8 m3
# MXN per truck trip
cost = {("North", "S1"): 900, ("North", "S2"): 1200, ("North", "S3"): 1500,
        ("South", "S1"): 1400, ("South", "S2"): 1000, ("South", "S3"): 800}

m = pulp.LpProblem("concrete_trucks", pulp.LpMinimize)
# x[p, s] = number of truck trips from plant p to site s (whole trucks)
x = {(p, s): m.add_variable(f"x_{p}_{s}", lowBound=0, cat="Integer")
     for p in plants for s in sites}

m += pulp.lpSum(cost[p, s] * x[p, s] for p in plants for s in sites)    # total cost
for p in plants:
    m += pulp.lpSum(x[p, s] for s in sites) <= trucks_per_day[p]       # plant capacity
for s in sites:
    m += TRUCK_M3 * pulp.lpSum(x[p, s] for p in plants) >= need_m3[s]  # demand, in m3
m += x["South", "S3"] <= 2                                             # bridge limit

result = m.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, "| cost =", pulp.value(m.objective))
for (p, s), v in x.items():
    if v.value() > 0.5:
        print(f"{p} -> {s}: {round(v.value())} trucks")

**Your turn: remove the twist.** In the cell above, make one change at a time, run it again and compare the optimum with the one you got first. Then undo the change.

* **trucks are continuous (plain LP)**: replace `cat="Integer")` with `cat="Continuous")`.
* **no bridge limit**: delete the line `m += x["South", "S3"] <= 2`.

If the optimum does not move, is the model still correct without that line? (The PDF discusses it.)

---
## Problem 1 · The snack box
**Class:** Diet problem (integer version, with variety logic)

### The card
The school cafeteria wants a cheap afternoon snack box. It may contain whole units of the items below, at most 2 units of each. The box must give at least 500 kcal, 20 g of protein and 8 g of fiber, and at most 35 g of sugar. For variety it must contain at least 4 different items, and at most one of the two sweet items (granola bar, cookie). What should go in the box?

| Item | Cost (MXN) | kcal | Protein (g) | Fiber (g) | Sugar (g) |
|---|---|---|---|---|---|
| Apple | 8 | 95 | 0.5 | 4.4 | 19 |
| Yogurt | 15 | 150 | 8 | 0 | 12 |
| Granola bar | 12 | 190 | 4 | 3 | 12 |
| Peanuts | 10 | 160 | 7 | 2.4 | 1 |
| Cheese stick | 11 | 80 | 7 | 0 | 0 |
| Cookie | 6 | 140 | 2 | 1 | 10 |
| Carrot sticks | 7 | 30 | 0.7 | 2 | 3 |

### The model
* **Sets:** items $i\in I$ (7 items); sweet items $S=\{\text{granola bar}, \text{cookie}\}$. **Data:** cost $c_i$, kcal $e_i$, protein $p_i$, fiber $f_i$, sugar $s_i$ per unit.
* **Variables:** $x_i\in\{0,1,2\}$ = units of item $i$; $y_i\in\{0,1\}$ = 1 if item $i$ is in the box.
* **Objective:** $\min \sum_i c_i x_i$ (MXN).

| | Constraint | In words |
|---|---|---|
| C1 | $\sum_i e_i x_i \ge 500,\ \sum_i p_i x_i \ge 20,\ \sum_i f_i x_i \ge 8$ | nutrition minimums |
| C2 | $\sum_i s_i x_i \le 35$ | sugar maximum |
| C3 | $x_i \le 2\,y_i \quad \forall i$ | no units of an item that is not in the box (and at most 2) |
| C4 | $x_i \ge y_i \quad \forall i$ | an item counted as in the box has at least one unit |
| C5 | $\sum_i y_i \ge 4$ | at least 4 different items |
| C6 | $y_{\text{granola}} + y_{\text{cookie}} \le 1$ | at most one sweet item |

In [ ]:
import pulp

items = ["Apple", "Yogurt", "Granola bar", "Peanuts", "Cheese stick", "Cookie",
         "Carrot sticks"]
cost  = dict(zip(items, [8, 15, 12, 10, 11, 6, 7]))           # MXN per unit
kcal  = dict(zip(items, [95, 150, 190, 160, 80, 140, 30]))
prot  = dict(zip(items, [0.5, 8, 4, 7, 7, 2, 0.7]))           # g
fiber = dict(zip(items, [4.4, 0, 3, 2.4, 0, 1, 2]))           # g
sugar = dict(zip(items, [19, 12, 12, 1, 0, 10, 3]))           # g

m = pulp.LpProblem("snack_box", pulp.LpMinimize)
# x[i] = units of item i (0, 1 or 2); y[i] = 1 if item i is in the box
x = {i: m.add_variable(f"x_{k}", lowBound=0, upBound=2, cat="Integer")
     for k, i in enumerate(items)}
y = {i: m.add_variable(f"y_{k}", cat="Binary") for k, i in enumerate(items)}

m += pulp.lpSum(cost[i] * x[i] for i in items)                 # total cost
m += pulp.lpSum(kcal[i] * x[i] for i in items) >= 500          # energy
m += pulp.lpSum(prot[i] * x[i] for i in items) >= 20           # protein
m += pulp.lpSum(fiber[i] * x[i] for i in items) >= 8           # fiber
m += pulp.lpSum(sugar[i] * x[i] for i in items) <= 35          # sugar
for i in items:
    m += x[i] <= 2 * y[i]          # no units unless the item is in the box
    m += x[i] >= y[i]              # an item "in the box" has at least 1 unit
m += pulp.lpSum(y[i] for i in items) >= 4                      # at least 4 different items
m += y["Granola bar"] + y["Cookie"] <= 1                       # at most one sweet item

result = m.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, "| cost =", pulp.value(m.objective))
for i in items:
    if x[i].value() > 0.5:
        print(i, round(x[i].value()))

**Your turn: remove the twist.** In the cell above, make one change at a time, run it again and compare the optimum with the one you got first. Then undo the change.

* **no x ≥ y link**: delete the line `m += x[i] >= y[i]`.
* **no variety rules (≥ 4 items, ≤ 1 sweet)**: delete the line `m += pulp.lpSum(y[i] for i in items) >= 4`; delete the line `m += y["Granola bar"] + y["Cookie"] <= 1`.

If the optimum does not move, is the model still correct without that line? (The PDF discusses it.)

---
## Problem 2 · Wi-Fi on campus
**Class:** Set covering

### The card
The campus IT office will install Wi-Fi antennas on some of 6 rooftops so that each of the 8 zones of the campus gets signal from at least one antenna. The library (zone 8) is crowded: it needs signal from at least two antennas. Rooftops R2 and R5 are on the same building and only one of them can be used. Which rooftops should get an antenna?

| Rooftop | Cost (k MXN) | Zones reached |
|---|---|---|
| R1 | 40 | 1, 2, 5 |
| R2 | 30 | 2, 3, 8 |
| R3 | 50 | 4, 5, 6, 8 |
| R4 | 35 | 1, 3, 7 |
| R5 | 25 | 6, 7, 8 |
| R6 | 45 | 3, 4, 7, 8 |

### The model
* **Sets:** rooftops $r\in\{R1,…,R6\}$, zones $z\in\{1,…,8\}$. **Data:** cost $c_r$; $a_{rz}=1$ if rooftop $r$ reaches zone $z$ (the table).
* **Variables:** $y_r\in\{0,1\}$ = 1 if an antenna is installed on rooftop $r$.
* **Objective:** $\min \sum_r c_r y_r$ (k MXN).
* **Constraints** ($\sum_r a_{rz} y_r \ge 1$ for every zone, written out):

| | Constraint | In words |
|---|---|---|
| C1.1 | $y_{R1} + y_{R4} \ge 1$ | zone 1 gets signal |
| C1.2 | $y_{R1} + y_{R2} \ge 1$ | zone 2 gets signal |
| C1.3 | $y_{R2} + y_{R4} + y_{R6} \ge 1$ | zone 3 gets signal |
| C1.4 | $y_{R3} + y_{R6} \ge 1$ | zone 4 gets signal |
| C1.5 | $y_{R1} + y_{R3} \ge 1$ | zone 5 gets signal |
| C1.6 | $y_{R3} + y_{R5} \ge 1$ | zone 6 gets signal |
| C1.7 | $y_{R4} + y_{R5} + y_{R6} \ge 1$ | zone 7 gets signal |
| C1.8 | $y_{R2} + y_{R3} + y_{R5} + y_{R6} \ge 2$ | library: two antennas |
| C2 | $y_{R2} + y_{R5} \le 1$ | same building: at most one of the two |

In [ ]:
import pulp

cost = {"R1": 40, "R2": 30, "R3": 50, "R4": 35, "R5": 25, "R6": 45}          # k MXN
reaches = {"R1": [1, 2, 5], "R2": [2, 3, 8], "R3": [4, 5, 6, 8],
           "R4": [1, 3, 7], "R5": [6, 7, 8], "R6": [3, 4, 7, 8]}
zones = range(1, 9)
antennas_needed = {z: 2 if z == 8 else 1 for z in zones}       # the library needs two

m = pulp.LpProblem("wifi", pulp.LpMinimize)
y = {r: m.add_variable(f"y_{r}", cat="Binary") for r in cost}  # 1 = antenna on rooftop r

m += pulp.lpSum(cost[r] * y[r] for r in cost)
for z in zones:
    # zone z must be reached by enough chosen rooftops
    m += pulp.lpSum(y[r] for r in cost if z in reaches[r]) >= antennas_needed[z]
m += y["R2"] + y["R5"] <= 1                       # same building: not both

result = m.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, "| cost =", pulp.value(m.objective))
print("Antennas on:", [r for r in cost if y[r].value() > 0.5])

**Your turn: remove the twist.** In the cell above, make one change at a time, run it again and compare the optimum with the one you got first. Then undo the change.

* **library covered once**: replace `2 if z == 8 else 1` with `1`.
* **R2 and R5 allowed together**: delete the line `m += y["R2"] + y["R5"] <= 1`.

If the optimum does not move, is the model still correct without that line? (The PDF discusses it.)

---
## Problem 3 · The morning routes
**Class:** Set partitioning

### The card
A bakery delivers to 7 cafés every morning. The dispatcher has already drawn 11 possible van routes; each route visits the cafés listed and has a fixed cost. Every café must be visited by exactly one van (two deliveries would waste bread), and the bakery has only 3 vans. Which routes should run?

| Route | Cafés visited | Cost (MXN x10) |
|---|---|---|
| A | 1, 2 | 30 |
| B | 3, 4, 5 | 42 |
| C | 6, 7 | 28 |
| D | 1, 3 | 26 |
| E | 2, 4, 6 | 45 |
| F | 5, 7 | 24 |
| G | 1, 2, 3, 4 | 70 |
| H | 4, 5, 6, 7 | 58 |
| I | 2, 5 | 22 |
| J | 6 | 15 |
| K | 4 | 12 |

### The model
* **Sets:** routes $r\in\{A,…,K\}$, cafés $c\in\{1,…,7\}$. **Data:** cost $c_r$; the cafés of each route (the table).
* **Variables:** $y_r\in\{0,1\}$ = 1 if route $r$ runs.
* **Objective:** $\min \sum_r c_r y_r$.
* **Constraints** ($\sum_{r \ni c} y_r = 1$ for every café, written out):

| | Constraint | In words |
|---|---|---|
| C1.1 | $y_A + y_D + y_G = 1$ | café 1: exactly one van |
| C1.2 | $y_A + y_E + y_G + y_I = 1$ | café 2: exactly one van |
| C1.3 | $y_B + y_D + y_G = 1$ | café 3: exactly one van |
| C1.4 | $y_B + y_E + y_G + y_H + y_K = 1$ | café 4: exactly one van |
| C1.5 | $y_B + y_F + y_H + y_I = 1$ | café 5: exactly one van |
| C1.6 | $y_C + y_E + y_H + y_J = 1$ | café 6: exactly one van |
| C1.7 | $y_C + y_F + y_H = 1$ | café 7: exactly one van |
| C2 | $\sum_r y_r \le 3$ | only 3 vans |

In [ ]:
import pulp

routes = {"A": [1, 2], "B": [3, 4, 5], "C": [6, 7], "D": [1, 3], "E": [2, 4, 6],
          "F": [5, 7], "G": [1, 2, 3, 4], "H": [4, 5, 6, 7], "I": [2, 5],
          "J": [6], "K": [4]}
cost = {"A": 30, "B": 42, "C": 28, "D": 26, "E": 45, "F": 24, "G": 70, "H": 58, "I": 22,
        "J": 15, "K": 12}
cafes = range(1, 8)

m = pulp.LpProblem("routes", pulp.LpMinimize)
y = {r: m.add_variable(f"y_{r}", cat="Binary") for r in routes}   # 1 = route r runs

m += pulp.lpSum(cost[r] * y[r] for r in routes)
for c in cafes:
    m += pulp.lpSum(y[r] for r in routes if c in routes[r]) == 1   # exactly one van
m += pulp.lpSum(y[r] for r in routes) <= 3                         # only 3 vans

result = m.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, "| cost =", pulp.value(m.objective))
print("Routes:", [r for r in routes if y[r].value() > 0.5])

**Your turn: remove the twist.** In the cell above, make one change at a time, run it again and compare the optimum with the one you got first. Then undo the change.

* **≥ 1 instead of = 1 (covering)**: replace `) == 1` with `) >= 1`.
* **no 3-van limit**: delete the line `m += pulp.lpSum(y[r] for r in routes) <= 3`.

If the optimum does not move, is the model still correct without that line? (The PDF discusses it.)

---
## Problem 4 · Warehouses for Monterrey
**Class:** Capacitated facility location (single sourcing)

### The card
A retailer may rent warehouses in Apodaca, Escobedo and Santa Catarina (monthly rent, capacity in pallets) to supply 5 stores. For simple paperwork, each store must be supplied by one single warehouse. Shipping costs are per pallet. Which warehouses should be rented, and which warehouse supplies each store?

| \$ per pallet | T1 | T2 | T3 | T4 | T5 | Rent (\$) | Capacity |
|---|---|---|---|---|---|---|---|
| Apodaca | 4 | 6 | 9 | 5 | 8 | 400 | 70 |
| Escobedo | 7 | 3 | 4 | 8 | 5 | 350 | 60 |
| Santa Catarina | 6 | 7 | 5 | 3 | 4 | 500 | 90 |
| Demand (pallets) | 30 | 25 | 20 | 35 | 15 |  |  |

### The model
* **Sets:** warehouses $w$, stores $t$. **Data:** rent $F_w$, capacity $K_w$, demand $d_t$, shipping cost per pallet $c_{wt}$.
* **Variables:** $y_w\in\{0,1\}$ = 1 if warehouse $w$ is rented; $x_{wt}\in\{0,1\}$ = 1 if $w$ supplies (all of) store $t$.
* **Objective:** $\min \sum_w F_w y_w + \sum_w\sum_t c_{wt}\,d_t\,x_{wt}$: rent + (cost per pallet $\times$ pallets).

| | Constraint | In words |
|---|---|---|
| C1 | $\sum_w x_{wt} = 1 \quad \forall t$ | every store has exactly one supplier |
| C2 | $\sum_t d_t\,x_{wt} \le K_w\,y_w \quad \forall w$ | capacity, and no shipments at all from a warehouse that is not rented |

In [ ]:
import pulp

wh = ["Apodaca", "Escobedo", "Santa Catarina"]
stores = ["T1", "T2", "T3", "T4", "T5"]
rent = {"Apodaca": 400, "Escobedo": 350, "Santa Catarina": 500}
cap = {"Apodaca": 70, "Escobedo": 60, "Santa Catarina": 90}                  # pallets
demand = {"T1": 30, "T2": 25, "T3": 20, "T4": 35, "T5": 15}                  # pallets
ship = {"Apodaca":        {"T1": 4, "T2": 6, "T3": 9, "T4": 5, "T5": 8},     # $ per pallet
        "Escobedo":       {"T1": 7, "T2": 3, "T3": 4, "T4": 8, "T5": 5},
        "Santa Catarina": {"T1": 6, "T2": 7, "T3": 5, "T4": 3, "T5": 4}}

m = pulp.LpProblem("warehouses", pulp.LpMinimize)
# y[w] = 1 if warehouse w is rented; x[w, t] = 1 if w supplies store t
y = {w: m.add_variable(f"y_{k}", cat="Binary") for k, w in enumerate(wh)}
x = {(w, t): m.add_variable(f"x_{k}_{t}", cat="Binary")
     for k, w in enumerate(wh) for t in stores}

m += pulp.lpSum(rent[w] * y[w] for w in wh) + \
     pulp.lpSum(ship[w][t] * demand[t] * x[w, t] for w in wh for t in stores)
for t in stores:
    m += pulp.lpSum(x[w, t] for w in wh) == 1                    # one supplier per store
for w in wh:
    # capacity, and no shipments at all from a warehouse that is not rented
    m += pulp.lpSum(demand[t] * x[w, t] for t in stores) <= cap[w] * y[w]

result = m.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, "| cost =", pulp.value(m.objective))
print("Rent:", [w for w in wh if y[w].value() > 0.5])
print("Supplier:", {t: w for w in wh for t in stores if x[w, t].value() > 0.5})

**Your turn: remove the twist.** In the cell above, make one change at a time, run it again and compare the optimum with the one you got first. Then undo the change.

* **split deliveries allowed (x continuous in [0, 1])**: replace `x = {(w, t): m.add_variable(f"x_{k}_{t}", cat="Binary")` with `x = {(w, t): m.add_variable(f"x_{k}_{t}", lowBound=0, upBound=1)`.
* **capacity not multiplied by y**: replace `<= cap[w] * y[w]` with `<= cap[w]`.

If the optimum does not move, is the model still correct without that line? (The PDF discusses it.)

---
## Problem 5 · The innovation fund
**Class:** Knapsack (multidimensional: two budgets) with logical constraints

### The card
A company chooses which of 7 innovation projects to fund. Each project is funded completely or not at all and needs money in year 1 and in year 2; the budgets are 80 k\$ in year 1 and 60 k\$ in year 2. Project B builds on project A (no B without A), and projects C and D compete for the same team (at most one of them). Which projects maximize the total NPV?

| Project | A | B | C | D | E | F | G |
|---|---|---|---|---|---|---|---|
| Year 1 (k\$) | 20 | 15 | 30 | 25 | 10 | 20 | 15 |
| Year 2 (k\$) | 10 | 20 | 10 | 15 | 25 | 5 | 20 |
| NPV (k\$) | 50 | 45 | 70 | 60 | 40 | 35 | 45 |

### The model
* **Data:** costs $a_p$ (year 1), $b_p$ (year 2), value $v_p$ (NPV).
* **Variables:** $y_p\in\{0,1\}$ = 1 if project $p$ is funded.
* **Objective:** $\max \sum_p v_p y_p$ (k\$).

| | Constraint | In words |
|---|---|---|
| C1 | $\sum_p a_p y_p \le 80$ | year-1 budget |
| C2 | $\sum_p b_p y_p \le 60$ | year-2 budget |
| C3 | $y_B \le y_A$ | no B without A (if $y_B=1$ then $y_A=1$) |
| C4 | $y_C + y_D \le 1$ | at most one of C and D |

In [ ]:
import pulp

P = ["A", "B", "C", "D", "E", "F", "G"]
year1 = dict(zip(P, [20, 15, 30, 25, 10, 20, 15]))       # k$
year2 = dict(zip(P, [10, 20, 10, 15, 25, 5, 20]))        # k$
npv   = dict(zip(P, [50, 45, 70, 60, 40, 35, 45]))       # k$

m = pulp.LpProblem("innovation_fund", pulp.LpMaximize)
y = {p: m.add_variable(f"y_{p}", cat="Binary") for p in P}      # 1 = fund project p

m += pulp.lpSum(npv[p] * y[p] for p in P)
m += pulp.lpSum(year1[p] * y[p] for p in P) <= 80               # year-1 budget
m += pulp.lpSum(year2[p] * y[p] for p in P) <= 60               # year-2 budget
m += y["B"] <= y["A"]                                           # no B without A
m += y["C"] + y["D"] <= 1                                       # C or D, not both

result = m.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, "| NPV =", pulp.value(m.objective))
print("Fund:", [p for p in P if y[p].value() > 0.5])

**Your turn: remove the twist.** In the cell above, make one change at a time, run it again and compare the optimum with the one you got first. Then undo the change.

* **no logic rules**: delete the line `m += y["B"] <= y["A"]`; delete the line `m += y["C"] + y["D"] <= 1`.
* **only the year-1 budget**: delete the line `m += pulp.lpSum(year2[p] * y[p] for p in P) <= 60`.

If the optimum does not move, is the model still correct without that line? (The PDF discusses it.)

---
## Problem 6 · Interns and projects
**Class:** Assignment (unbalanced)

### The card
Five interns join for the summer and there are 6 projects. Each intern works on exactly one project and each project gets at most one intern, so one project will wait. Project P6 is a client commitment and must be staffed. The table gives the hours each intern would need (— means the intern lacks the skills). Which assignment minimizes the total hours?

| Hours | P1 | P2 | P3 | P4 | P5 | P6 |
|---|---|---|---|---|---|---|
| Ana | 9 | 12 | — | 8 | 11 | 10 |
| Bruno | 10 | 9 | 12 | 13 | 8 | 11 |
| Carla | 7 | 13 | 10 | 9 | 12 | 14 |
| Diego | 11 | 8 | 9 | 10 | 14 | 12 |
| Elena | — | 10 | 8 | 11 | 9 | 13 |

### The model
* **Sets:** interns $i$, projects $j$; allowed pairs $A$ (all pairs except Ana-P3 and Elena-P1). **Data:** hours $h_{ij}$.
* **Variables:** $x_{ij}\in\{0,1\}$ for $(i,j)\in A$ = 1 if intern $i$ works on project $j$ (no variable for a forbidden pair).
* **Objective:** $\min \sum_{(i,j)\in A} h_{ij} x_{ij}$ (hours).

| | Constraint | In words |
|---|---|---|
| C1 | $\sum_j x_{ij} = 1 \quad \forall i$ | each intern: exactly one project |
| C2 | $\sum_i x_{ij} \le 1 \quad \forall j$ | each project: at most one intern |
| C3 | $\sum_i x_{i,P6} = 1$ | P6 must be staffed |

In [ ]:
import pulp

interns = ["Ana", "Bruno", "Carla", "Diego", "Elena"]
projects = ["P1", "P2", "P3", "P4", "P5", "P6"]
H = [[9, 12, None, 8, 11, 10],        # hours; None = lacks the skills
     [10, 9, 12, 13, 8, 11],
     [7, 13, 10, 9, 12, 14],
     [11, 8, 9, 10, 14, 12],
     [None, 10, 8, 11, 9, 13]]
hours = {(i, j): H[a][b] for a, i in enumerate(interns) for b, j in enumerate(projects)
         if H[a][b] is not None}                        # allowed pairs only

m = pulp.LpProblem("interns", pulp.LpMinimize)
# x[i, j] = 1 if intern i works on project j (no variable for a forbidden pair)
x = {(i, j): m.add_variable(f"x_{i}_{j}", cat="Binary") for (i, j) in hours}

m += pulp.lpSum(hours[k] * x[k] for k in x)
for i in interns:
    m += pulp.lpSum(x[i, j] for j in projects if (i, j) in x) == 1    # exactly one project
for j in projects:
    m += pulp.lpSum(x[i, j] for i in interns if (i, j) in x) <= 1     # at most one intern
m += pulp.lpSum(x[i, "P6"] for i in interns if (i, "P6") in x) == 1   # P6 is staffed

result = m.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, "| hours =", pulp.value(m.objective))
print({i: j for (i, j) in x if x[i, j].value() > 0.5})

**Your turn: remove the twist.** In the cell above, make one change at a time, run it again and compare the optimum with the one you got first. Then undo the change.

* **P6 not mandatory**: delete the line `m += pulp.lpSum(x[i, "P6"] for i in interns if (i, "P6") in x) == 1`.
* **= 1 for projects too**: replace `if (i, j) in x) <= 1` with `if (i, j) in x) == 1`.

If the optimum does not move, is the model still correct without that line? (The PDF discusses it.)

---
## Problem 7 · The night premium
**Class:** Staff (shift) scheduling: a covering problem with general integers

### The card
A hospital ward splits the day into six 4-hour blocks. A nurse works 8 hours in a row (two consecutive blocks) and can start at the beginning of any block; a shift starting at 20:00 ends at 04:00. Each block needs a minimum number of nurses. Shifts that start at night cost more. How many nurses should start at each time to cover the ward at minimum cost?

| Block | 00-04 | 04-08 | 08-12 | 12-16 | 16-20 | 20-24 |
|---|---|---|---|---|---|---|
| Nurses needed | 5 | 7 | 12 | 11 | 9 | 6 |
| Cost of a shift starting here (MXN) | 1500 | 1200 | 1000 | 1000 | 1200 | 1500 |

### The model
* **Sets:** blocks $b=1,…,6$ (00-04, …, 20-24). **Data:** need $n_b$, cost $c_b$ of a shift starting in block $b$.
* **Variables:** $x_b\in\mathbb{Z}_{\ge 0}$ = number of nurses starting at block $b$.
* **Objective:** $\min \sum_b c_b x_b$ (MXN).
* **Constraints:** on duty in block $b$ = started in $b$ or in $b-1$, with block 0 = block 6 (wrap-around midnight):

| | Constraint | In words |
|---|---|---|
| C1.1 | $x_{1} + x_{6} \ge 5$ | block 00-04 (the 20:00 starters are still on duty) |
| C1.2 | $x_{2} + x_{1} \ge 7$ | block 04-08 |
| C1.3 | $x_{3} + x_{2} \ge 12$ | block 08-12 |
| C1.4 | $x_{4} + x_{3} \ge 11$ | block 12-16 |
| C1.5 | $x_{5} + x_{4} \ge 9$ | block 16-20 |
| C1.6 | $x_{6} + x_{5} \ge 6$ | block 20-24 |
| C2 | $x_b \in \{0,1,2,\dots\}$ | whole nurses |

In [ ]:
import pulp

blocks = ["00-04", "04-08", "08-12", "12-16", "16-20", "20-24"]
need = [5, 7, 12, 11, 9, 6]                                  # nurses needed in each block
cost = [1500, 1200, 1000, 1000, 1200, 1500]                  # MXN per shift, by start block
B = range(6)

m = pulp.LpProblem("nurses", pulp.LpMinimize)
# x[b] = number of nurses starting at block b
x = {b: m.add_variable(f"x_{b}", lowBound=0, cat="Integer") for b in B}

m += pulp.lpSum(cost[b] * x[b] for b in B)
for b in B:
    # on duty in block b = started in b or one block before;
    # (b - 1) % 6 wraps block 00-04 back to the 20-24 start
    m += x[b] + x[(b - 1) % 6] >= need[b]

result = m.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, "| cost =", pulp.value(m.objective))
print({blocks[b]: round(x[b].value()) for b in B})

**Your turn: remove the twist.** In the cell above, make one change at a time, run it again and compare the optimum with the one you got first. Then undo the change.

* **nurses continuous (plain LP)**: replace `lowBound=0, cat="Integer")` with `lowBound=0)`.
* **no wrap-around midnight**: replace `x[b] + x[(b - 1) % 6] >= need[b]` with `x[b] + (x[b - 1] if b > 0 else 0) >= need[b]`.

If the optimum does not move, is the model still correct without that line? (The PDF discusses it.)

---
## Problem 8 · Loading the vans
**Class:** Bin packing (two capacities)

### The card
Ten parcels leave the warehouse today. A van carries at most 100 kg and, because of its shelves, at most 3 parcels. Parcels 1 and 2 are fragile and cannot travel in the same van. What is the minimum number of vans, and which parcels go in each one?

| Parcel | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| Weight (kg) | 55 | 45 | 40 | 35 | 30 | 30 | 25 | 20 | 15 | 10 |

### The model
* **Sets:** parcels $i=1,…,10$; candidate vans $k=1,…,10$ (never more vans than parcels). **Data:** weight $w_i$.
* **Variables:** $y_k\in\{0,1\}$ = 1 if van $k$ is used; $x_{ik}\in\{0,1\}$ = 1 if parcel $i$ goes in van $k$.
* **Objective:** $\min \sum_k y_k$ (number of vans).

| | Constraint | In words |
|---|---|---|
| C1 | $\sum_k x_{ik} = 1 \quad \forall i$ | every parcel goes in exactly one van |
| C2 | $\sum_i w_i x_{ik} \le 100\,y_k \quad \forall k$ | weight, and only in a used van |
| C3 | $\sum_i x_{ik} \le 3\,y_k \quad \forall k$ | shelves: at most 3 parcels |
| C4 | $x_{1k} + x_{2k} \le 1 \quad \forall k$ | fragile parcels 1 and 2 apart |
| C5 | $y_k \le y_{k-1} \quad k\ge 2$ | optional: use vans in order (same optimum, faster) |

In [ ]:
import pulp

w = [55, 45, 40, 35, 30, 30, 25, 20, 15, 10]       # kg, parcels 1..10
n = len(w)
N = range(n)          # parcels (index 0 = parcel 1)
K = range(n)          # candidate vans: never more vans than parcels

m = pulp.LpProblem("vans", pulp.LpMinimize)
# y[k] = 1 if van k is used; x[i, k] = 1 if parcel i goes in van k
y = {k: m.add_variable(f"y_{k}", cat="Binary") for k in K}
x = {(i, k): m.add_variable(f"x_{i}_{k}", cat="Binary") for i in N for k in K}

m += pulp.lpSum(y[k] for k in K)                                     # number of vans
for i in N:
    m += pulp.lpSum(x[i, k] for k in K) == 1                  # in exactly one van
for k in K:
    m += pulp.lpSum(w[i] * x[i, k] for i in N) <= 100 * y[k]  # weight, only if used
    m += pulp.lpSum(x[i, k] for i in N) <= 3 * y[k]           # at most 3 parcels
    m += x[0, k] + x[1, k] <= 1                               # fragile 1 and 2 apart
for k in K[1:]:
    m += y[k] <= y[k - 1]          # optional: use vans in order (same optimum, faster)

result = m.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, "| vans =", pulp.value(m.objective))
for k in K:
    if y[k].value() > 0.5:
        print("van", k + 1, [i + 1 for i in N if x[i, k].value() > 0.5])

**Your turn: remove the twist.** In the cell above, make one change at a time, run it again and compare the optimum with the one you got first. Then undo the change.

* **no 3-parcel limit**: delete the line `m += pulp.lpSum(x[i, k] for i in N) <= 3 * y[k]`.
* **fragile pair allowed together**: delete the line `m += x[0, k] + x[1, k] <= 1`.

If the optimum does not move, is the model still correct without that line? (The PDF discusses it.)